# Heatmap della modulazione (max - min di CH2) sulla griglia - versione con smoothing

Legge i file di un run di `acquisizione_matrice_xy.ipynb` o `acquisizione_griglia_xy.ipynb` (la cartella `.../run_00N` con `griglia.csv` e un CSV per punto) e, per ogni punto, calcola **max - min del canale 2 solo nella zona in cui viene inviata la scansione**, con **smoothing**: CH2 nella zona viene (opzionalmente) liberato dai picchi positivi stretti con un'apertura (minimo mobile + massimo mobile, `PRE_APERTURA`), ridotto a `N_PUNTI` con la media a blocchi e smussato con il metodo scelto in `METODO` (`savgol`, `gaussiano` o `media_mobile`); il max - min si prende sulla curva smussata, cioè sulle campane. Gli stessi parametri sono in `smoothing_curva.ipynb`, dove puoi provare il metodo su un singolo punto prima di lanciare la heatmap. Con picchi negativi usa invece `heatmap_modulazione.ipynb` (mediana a blocchi), che tratta i picchi di entrambi i segni.

**Come si trova la zona della scansione (da CH1, la rampa del laser):** inizia dove CH1 supera per la prima volta `V_INIZIO_V` (0.9 V, il livello del trigger) e finisce dove CH1 raggiunge il suo massimo (fine della rampa in salita, prima del ritorno). Quello che sta prima (plateau iniziale) e dopo (ritorno del laser) viene ignorato.

Ordine: cella 1 (scrivi la cartella del run in `CARTELLA_RUN` e controlla le altre impostazioni), 2 (legge il registro del run), 3 (calcolo; con 64 punti da 240 MB può durare qualche minuto, i risultati vengono salvati in `heatmap_valori.csv` e riusati), 4 (controllo di un punto: **la curva nera deve seguire le campane senza i picchi (se le appiattisce, riduci le larghezze dei filtri; se restano picchi, aumentale): guardala prima di fidarti della heatmap**), 5 (heatmap).

In [ ]:
# Cella 1 - impostazioni
import csv
import os
import time

import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d, maximum_filter1d, minimum_filter1d, uniform_filter1d
from scipy.signal import savgol_filter

try:
    import pandas as pd
except ImportError:
    pd = None  # senza pandas la lettura dei CSV da 240 MB e' molto piu' lenta

CARTELLA_RUN = ""          # <-- SCRIVI QUI la cartella del run, es. "2026-10-05/run_002" (relativa al notebook) o un percorso completo
V_INIZIO_V = 0.9           # CH1 sopra questo valore = inizio della scansione (livello del trigger)
TOLLERANZA_FINE_V = 0.3    # la rampa e' finita quando CH1 e' a meno di questo dal suo massimo
PRE_APERTURA = True        # True: toglie i picchi spuri POSITIVI stretti (minimo mobile + massimo mobile) prima dello smoothing
APERTURA_CAMPIONI = 50     # larghezza dell'apertura in campioni: poco piu' larga dei picchi e molto piu' stretta delle campane
N_PUNTI = 20000            # la zona viene ridotta a questo numero di punti (media a blocchi) prima dello smoothing
METODO = "savgol"          # "savgol" | "gaussiano" | "media_mobile"
SAVGOL_FINESTRA = 201      # punti della curva ridotta (dispari)
SAVGOL_ORDINE = 3
GAUSS_SIGMA = 40           # punti della curva ridotta
MEDIA_FINESTRA = 101       # punti della curva ridotta
PERCENTILI = (0, 100)      # sulla curva smussata: (0, 100) = max - min esatto
RICALCOLA = False          # True: ignora heatmap_valori_smoothing.csv e rilegge tutti i CSV (da usare dopo aver cambiato i parametri)
ASSI_RELATIVI_UM = True    # True: assi in um rispetto al centro della matrice; False: posizioni in mm
PUNTO_CONTROLLO = 0        # indice del punto mostrato nella cella 4

In [ ]:
# Cella 2 - legge il registro dei punti del run scelto
if not CARTELLA_RUN.strip():
    raise ValueError("Scrivi in cella 1 la cartella del run in CARTELLA_RUN, es. '2026-10-05/run_002'")
run_folder = os.path.abspath(CARTELLA_RUN)   # i percorsi relativi partono dalla cartella del notebook
registro_path = os.path.join(run_folder, "griglia.csv")
if not os.path.isfile(registro_path):
    raise FileNotFoundError(f"griglia.csv non trovato in {run_folder}: controlla CARTELLA_RUN")

with open(registro_path, newline="") as f:
    registro = list(csv.DictReader(f))
print(f"Run: {run_folder}")
print(f"{len(registro)} punti nel registro")

In [ ]:
# Cella 3 - per ogni punto: zona della scansione (da CH1) e max-min di CH2 in quella zona
def carica_canali(percorso):
    if pd is not None:
        df = pd.read_csv(percorso, dtype=np.float32)
        return df["CH1"].to_numpy(), df["CH2"].to_numpy()
    dati = np.loadtxt(percorso, delimiter=",", skiprows=1, dtype=np.float32)
    return dati[:, 0], dati[:, 1]


def zona_scansione(ch1):
    """Indici (i0, i1) della rampa in salita di CH1, oppure None se non c'e' nessuna scansione."""
    sopra = np.flatnonzero(ch1 > V_INIZIO_V)
    if sopra.size == 0:
        return None
    i0 = int(sopra[0])
    fine = np.flatnonzero(ch1[i0:] >= ch1.max() - TOLLERANZA_FINE_V)
    i1 = i0 + int(fine[0])
    return (i0, i1) if i1 > i0 else None


def segnale_pulito(ch2, i0, i1):
    """CH2 nella zona della scansione smussato: (apertura) -> media a blocchi -> smoothing.
    Restituisce (curva smussata, campioni per blocco)."""
    seg = ch2[i0:i1 + 1].astype(np.float32)
    if PRE_APERTURA:
        w = max(3, int(APERTURA_CAMPIONI))
        seg = maximum_filter1d(minimum_filter1d(seg, size=w, mode="nearest"), size=w, mode="nearest")
    n = max(1, len(seg) // int(N_PUNTI))
    k = len(seg) // n
    ridotta = seg[:k * n].astype(np.float64).reshape(k, n).mean(axis=1)
    if METODO == "savgol":
        w = int(SAVGOL_FINESTRA) | 1
        w = min(w, k if k % 2 == 1 else k - 1)
        liscia = savgol_filter(ridotta, window_length=w, polyorder=int(SAVGOL_ORDINE), mode="nearest")
    elif METODO == "gaussiano":
        liscia = gaussian_filter1d(ridotta, sigma=float(GAUSS_SIGMA), mode="nearest")
    elif METODO == "media_mobile":
        liscia = uniform_filter1d(ridotta, size=int(MEDIA_FINESTRA), mode="nearest")
    else:
        raise ValueError("METODO deve essere 'savgol', 'gaussiano' o 'media_mobile'")
    return liscia, n


def modulazione(ch2, i0, i1):
    curva, _ = segnale_pulito(ch2, i0, i1)
    lo, hi = np.percentile(curva, PERCENTILI)
    return float(hi - lo)


valori_path = os.path.join(run_folder, "heatmap_valori_smoothing.csv")
risultati = []

if os.path.isfile(valori_path) and not RICALCOLA:
    with open(valori_path, newline="") as f:
        for r in csv.DictReader(f):
            risultati.append((int(r["indice"]), float(r["x_mm"]), float(r["y_mm"]),
                              int(r["i0"]), int(r["i1"]), float(r["modulazione_V"])))
    print(f"Riusati {len(risultati)} risultati da {valori_path} (se cambi METODO, le larghezze dei filtri o le soglie, metti RICALCOLA = True)")
else:
    t0 = time.time()
    for r in registro:
        percorso = os.path.join(run_folder, r["file"])
        if not os.path.isfile(percorso):
            print(f"  punto {r['indice']}: file mancante ({r['file']}), salto")
            continue
        ch1, ch2 = carica_canali(percorso)
        zona = zona_scansione(ch1)
        if zona is None:
            print(f"  punto {r['indice']}: nessuna scansione trovata in CH1 (mai sopra {V_INIZIO_V} V)")
            risultati.append((int(r["indice"]), float(r["x_mm"]), float(r["y_mm"]), -1, -1, float("nan")))
            continue
        i0, i1 = zona
        risultati.append((int(r["indice"]), float(r["x_mm"]), float(r["y_mm"]), i0, i1, modulazione(ch2, i0, i1)))
        print(f"  punto {r['indice']}: scansione {i0}..{i1} ({(i1 - i0) / len(ch1) * 100:.0f}% del record), "
              f"max-min CH2 = {risultati[-1][5]:.4f} V  [{time.time() - t0:.0f} s]")
    with open(valori_path, "w") as f:
        f.write("indice,x_mm,y_mm,i0,i1,modulazione_V\n")
        for riga in risultati:
            f.write(f"{riga[0]},{riga[1]:.4f},{riga[2]:.4f},{riga[3]},{riga[4]},{riga[5]:.6f}\n")
    print(f"Salvato: {valori_path}")

In [ ]:
# Cella 4 - CONTROLLO su un punto: la zona evidenziata deve essere la rampa di CH1
riga = next((r for r in risultati if r[0] == PUNTO_CONTROLLO), None)
if riga is None:
    raise ValueError(f"Punto {PUNTO_CONTROLLO} non trovato")
indice, x, y, i0, i1, mod = riga
nome_file = next(r["file"] for r in registro if int(r["indice"]) == indice)
ch1, ch2 = carica_canali(os.path.join(run_folder, nome_file))

passo = max(1, len(ch1) // 5000)   # decimazione solo per il disegno
asse = np.arange(len(ch1))[::passo]
fig, (a1, a2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
a1.plot(asse, ch1[::passo], color="tab:red", lw=1)
a1.axhline(V_INIZIO_V, color="k", ls="--", lw=0.8)
a1.set_ylabel("CH1 (V)")
a2.plot(asse, ch2[::passo], color="tab:blue", lw=0.6, alpha=0.5, label="CH2 grezzo")
if i0 >= 0:
    curva, n = segnale_pulito(ch2, i0, i1)
    a2.plot(i0 + (np.arange(len(curva)) + 0.5) * n, curva, color="k", lw=1.4, label="CH2 smussato")
    a2.legend(loc="best")
a2.set_ylabel("CH2 (V)")
a2.set_xlabel("indice del campione")
if i0 >= 0:
    for a in (a1, a2):
        a.axvspan(i0, i1, color="orange", alpha=0.25)
a1.set_title(f"Punto {indice}: X={x:.4f} mm, Y={y:.4f} mm | max-min CH2 (smoothing, nella zona) = {mod:.4f} V")
plt.tight_layout()
plt.show()

In [ ]:
# Cella 5 - heatmap di max - min di CH2 (solo zona della scansione)
x = np.array([r[1] for r in risultati])
y = np.array([r[2] for r in risultati])
z = np.array([r[5] for r in risultati])

x0, y0 = (x.mean(), y.mean()) if ASSI_RELATIVI_UM else (0.0, 0.0)
fattore = 1000.0 if ASSI_RELATIVI_UM else 1.0
unita = "um dal centro" if ASSI_RELATIVI_UM else "mm"
xp, yp = (x - x0) * fattore, (y - y0) * fattore

xs, ys = np.unique(np.round(xp, 3)), np.unique(np.round(yp, 3))
uniforme = (
    len(xs) > 1 and len(ys) > 1 and len(xs) * len(ys) == len(z)
    and np.allclose(np.diff(xs), np.diff(xs).mean(), rtol=1e-2)
    and np.allclose(np.diff(ys), np.diff(ys).mean(), rtol=1e-2)
)

fig, ax = plt.subplots(figsize=(7, 6))
if uniforme:
    Z = np.full((len(ys), len(xs)), np.nan)
    for xi, yi, zi in zip(np.round(xp, 3), np.round(yp, 3), z):
        Z[np.searchsorted(ys, yi), np.searchsorted(xs, xi)] = zi
    dx, dy = np.diff(xs).mean(), np.diff(ys).mean()
    im = ax.imshow(Z, origin="lower", aspect="equal", cmap="viridis",
                   extent=[xs[0] - dx / 2, xs[-1] + dx / 2, ys[0] - dy / 2, ys[-1] + dy / 2])
else:
    im = ax.scatter(xp, yp, c=z, s=400, cmap="viridis", marker="s")
    ax.set_aspect("equal")
fig.colorbar(im, ax=ax, label="max - min di CH2 nella zona della scansione (V)")
ax.set_xlabel(f"X ({unita})")
ax.set_ylabel(f"Y ({unita})")
ax.set_title("Modulazione CH2 sulla griglia")
plt.tight_layout()
fig.savefig(os.path.join(run_folder, "heatmap_modulazione_smoothing.png"), dpi=200)
plt.show()
print(f"Salvata anche in {os.path.join(run_folder, 'heatmap_modulazione_smoothing.png')}")